# VisionGuard training and DagsHub tracking

This notebook runs the reproducible audit, split generation, training, and MLflow logging workflow in Google Colab. Add `DAGSHUB_USER` and `DAGSHUB_TOKEN` as Colab Secrets before running the tracking cell. Never place the token in notebook code or screenshots.

In [ ]:
!git clone https://github.com/abdul-raheem-fast/23L-0588_A2_Ops.git
%cd 23L-0588_A2_Ops
!pip install -q -r requirements.txt kagglehub
import os
import random
import numpy as np
import tensorflow as tf
print('TensorFlow:', tf.__version__)
print('GPU devices:', tf.config.list_physical_devices('GPU'))
SEED = 23
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

In [ ]:
import kagglehub
dataset_path = kagglehub.dataset_download('cashbowman/ai-generated-images-vs-real-images')
print('Path to dataset files:', dataset_path)
from pathlib import Path
for path in sorted(Path(dataset_path).iterdir()):
    print(path, 'files:', sum(1 for item in path.rglob('*') if item.is_file()))

In [ ]:
from pathlib import Path
from scripts.audit_dataset import audit
from scripts.make_splits import main as make_splits_main
import sys
audit_output = Path('artifacts/audit')
audit_output.mkdir(parents=True, exist_ok=True)
rows = audit(dataset_path)
import csv, json
with (audit_output / 'image_audit.csv').open('w', newline='', encoding='utf-8') as file:
    writer = csv.DictWriter(file, fieldnames=rows[0].keys())
    writer.writeheader()
    writer.writerows(rows)
print('Audited images:', len(rows))
sys.argv = ['make_splits', '--dataset-dir', dataset_path, '--output-dir', 'artifacts/splits', '--seed', '23']
make_splits_main()

In [ ]:
from google.colab import userdata
import os
os.environ['DAGSHUB_USER'] = userdata.get('DAGSHUB_USER')
os.environ['DAGSHUB_TOKEN'] = userdata.get('DAGSHUB_TOKEN')
os.environ['MLFLOW_TRACKING_USERNAME'] = os.environ['DAGSHUB_USER']
os.environ['MLFLOW_TRACKING_PASSWORD'] = os.environ['DAGSHUB_TOKEN']
os.environ['MLFLOW_TRACKING_URI'] = f"https://dagshub.com/{os.environ['DAGSHUB_USER']}/23L-0588_A2_Ops.mlflow"
os.environ['MLFLOW_EXPERIMENT'] = 'VisionGuard'
import mlflow
mlflow.set_tracking_uri(os.environ['MLFLOW_TRACKING_URI'])
print('DagsHub MLflow URI configured for user:', os.environ['DAGSHUB_USER'])

In [ ]:
import json
from pathlib import Path
config = json.loads(Path('configs/visionguard.json').read_text())
config.update({'data_dir': 'artifacts/splits', 'epochs': 10, 'batch_size': 16, 'seed': 23})
Path('configs/colab.json').write_text(json.dumps(config, indent=2))
!python scripts/train.py --config configs/colab.json

In [ ]:
from google.colab import files
from pathlib import Path
import shutil
candidates = list(Path('models').glob('*.keras'))
print('Saved models:', [str(path) for path in candidates])
best = Path('models/mobilenetv2_finetuned.keras')
if best.exists():
    Path('inference-service/models').mkdir(parents=True, exist_ok=True)
    shutil.copy2(best, 'inference-service/models/champion.keras')
    print('Champion copied to inference-service/models/champion.keras')
shutil.make_archive('visionguard-training-artifacts', 'zip', 'artifacts')
shutil.make_archive('visionguard-models', 'zip', 'models')
files.download('visionguard-training-artifacts.zip')
files.download('visionguard-models.zip')